# Tạo Bộ Dữ Liệu Sạch & Làm Giàu Biến Thể Excel Thật (`BuildDataset.ipynb`)

Notebook này thực hiện trọn gói 3 bước:
1. **Giữ nguyên 100% file gốc trong `Data/Raw/`** và **làm sạch triệt để nhãn biên** (tự động loại bỏ các dòng ghi chú/chân trang và dòng tiêu đề biểu bị gắn nhầm vào vùng bảng ở bản cũ).
2. **Sinh tập file `.xlsx` biến thể thật vào `Data/Raw_Variants/`** (14 kiểu biến thể thực tế: bỏ ô gộp, không kẻ khung, xóa dòng mã cột, số dạng chuỗi, dòng trống xen kẽ, chân trang dài, dịch vị trí, thêm cột ghi chú, 2 bảng trên 1 sheet, nhiều sheet kèm sheet hướng dẫn không có bảng, đảo cột, lệch cột kiểu 11b và 2 tổ hợp khó), cân bằng trên toàn bộ 28 loại biểu mẫu và **chia đúng split của file gốc (chống rò rỉ dữ liệu tuyệt đối)**.
3. **Xuất bộ dữ liệu sạch chuẩn hóa** `Data/dataset.jsonl` + `Data/dataset_report.json` kèm kiểm tra toàn vẹn và xem trước trực quan cả file gốc lẫn file biến thể.

Chỉ cần chọn **Run All** (chạy được trên mọi Python 3.10+ có `openpyxl`).

In [1]:
from pathlib import Path
import json
import sys

def find_module_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'dataset_builder.py').is_file() and (candidate / 'Data' / 'Raw').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy dataset_builder.py và Data/Raw.')

MODULE_ROOT = find_module_root()
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

# ================= CẤU HÌNH SINH DỮ LIỆU =================
GENERATE_RAW_VARIANTS = True   # True: sinh lại toàn bộ file .xlsx biến thể trong Data/Raw_Variants/
MAX_TRAIN_SEEDS = 85           # Số workbook gốc đa dạng (trên 28 biểu) dùng sinh 14 biến thể cho train
MAX_VAL_SEEDS = 14             # Số workbook gốc dùng sinh biến thể đánh giá cho validation
MAX_TEST_ID_SEEDS = 20         # Số workbook gốc dùng sinh biến thể đánh giá cho test_id
MAX_TEST_OOD_SEEDS = 16        # Số workbook gốc dùng sinh biến thể đánh giá cho test_ood
SEED = 20260913
# =========================================================

CURATED_LABELS = [
    {
        'source_file': 'Data/Raw/Sở Giáo dục và đào tạo_21a_TP_HTPLDN_Kết quả triển khai công tác hỗ trợ phá(...).xlsx',
        'sheet_index': 0,
        'tables': [
            {'top': 2, 'left': 1, 'bottom': 8, 'right': 13, 'header_bottom': 7}
        ],
    }
]

import openpyxl
print('Module root:', MODULE_ROOT)
print('Python     :', sys.version.split()[0], '| openpyxl:', openpyxl.__version__)

Module root: d:\HoangTV\eform_btp1\eform_btp\Module\AI Import
Python     : 3.10.11 | openpyxl: 3.1.5


## Bước 1: Sinh các file Excel biến thể thật (`Data/Raw_Variants/`)

Mỗi file `.xlsx` sinh ra đều có thể mở trực tiếp bằng Excel để kiểm tra. Mọi biến thể kế thừa đúng `split` và `layout_group` của file gốc, bảo đảm không rò rỉ giữa `train`, `validation`, `test_id` và `test_ood`.

In [2]:
from raw_variant_generator import generate_raw_variants

if GENERATE_RAW_VARIANTS:
    variant_summary = generate_raw_variants(
        MODULE_ROOT,
        curated_labels=CURATED_LABELS,
        max_train_seeds=MAX_TRAIN_SEEDS,
        max_val_seeds=MAX_VAL_SEEDS,
        max_test_id_seeds=MAX_TEST_ID_SEEDS,
        max_test_ood_seeds=MAX_TEST_OOD_SEEDS,
        seed=SEED,
        clean_existing=True,
        progress=True,
    )
    print(json.dumps(variant_summary, ensure_ascii=False, indent=2))
else:
    print('Giữ nguyên Data/Raw_Variants/ hiện có.')

Hoàn tất sinh biến thể: 1394 file .xlsx (1500 sheet) -> Data/Raw_Variants
  train: 1195 | validation: 82 | test_id: 120 | test_ood: 103


## Bước 2: Làm sạch nhãn gốc & hợp nhất thành `Data/dataset.jsonl` + `Data/dataset_report.json`

In [3]:
from dataset_builder import BuildOptions, build_dataset

OPTIONS = BuildOptions(
    module_root=MODULE_ROOT,
    augment=True,
    include_raw_variants=True,
    minimum_family_samples=12,
    maximum_augmentations_per_sheet=5,
)
report = build_dataset(OPTIONS, curated_labels=CURATED_LABELS, progress=True)

assert report['ready_for_training'], report['parse_errors']
assert report['validation']['valid'], report['validation']
assert report['validation']['test_ood_has_no_family_leakage']
assert report['validation']['contains_no_doctype_field']
assert report['validation']['contains_no_raw_cell_values']

print(json.dumps({
    'ready_for_training': report['ready_for_training'],
    'schema_version': report['schema_version'],
    'raw_workbooks': report['raw_workbook_count'],
    'unique_raw_workbooks': report['unique_workbook_count'],
    'raw_variant_sheets': report['raw_variant_sheet_count'],
    'total_sheet_samples': report['sheet_sample_count'],
    'total_tables': report['table_count'],
    'no_table_sheets': report['no_table_sheet_count'],
    'multi_table_sheets': report['multi_table_sheet_count'],
    'col_mismatch_sheets': report['col_mismatch_sheet_count'],
    'label_cleaning_summary': report['label_cleaning_summary'],
    'splits': report['split_sheet_counts'],
    'variant_type_counts': report['variant_type_counts'],
}, ensure_ascii=False, indent=2))

{
  "ready_for_training": true,
  "schema_version": "excel-table-regions-2.0",
  "raw_workbooks": 1232,
  "unique_raw_workbooks": 929,
  "raw_variant_sheets": 1500,
  "total_sheet_samples": 2429,
  "total_tables": 2456,
  "no_table_sheets": 106,
  "multi_table_sheets": 133,
  "col_mismatch_sheets": 60,
  "label_cleaning_summary": {
    "top_trimmed": 0,
    "bottom_trimmed": 12,
    "cols_trimmed": 0
  },
  "splits": {
    "test_id": 250,
    "test_ood": 133,
    "train": 1894,
    "validation": 152
  },
  "variant_type_counts": {
    "blank_rows": 62,
    "borderless": 107,
    "col_mismatch": 60,
    "combo_nocode_numtext_shifted": 105,
    "combo_unmerged_borderless_footer": 105,
    "extra_note_col": 107,
    "long_footer": 107,
    "multi_sheet_data": 106,
    "no_code_row": 107,
    "no_table_guide_sheet": 106,
    "numeric_text": 102,
    "original": 929,
    "shifted_pos": 107,
    "swapped_cols": 106,
    "two_tables": 106,
    "unmerged": 107
  }
}


## Bước 3: Kiểm tra trực quan nhãn sạch (file gốc & các biến thể đã sinh)

In [4]:
from html import escape
from IPython.display import HTML, display
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.utils.cell import range_boundaries

records = [
    json.loads(line)
    for line in (MODULE_ROOT / 'Data' / 'dataset.jsonl').read_text(encoding='utf-8').splitlines()
    if line.strip()
]

def show_table_preview(record, table_index=0, max_rows=10, max_columns=14):
    if not record['tables']:
        display(HTML(f"<h4>{escape(record['split'])} · {escape(record['variant_type'])} · KHÔNG CÓ BẢNG</h4><p>{escape(record['source_file'])}</p>"))
        return
    wb = load_workbook(MODULE_ROOT / record['source_file'], data_only=False, read_only=False)
    try:
        sheet = wb.worksheets[record['sheet_index']]
        table = record['tables'][table_index]
        visible_bottom = min(table['bottom'], table['top'] + max_rows - 1)
        visible_right = min(table['right'], table['left'] + max_columns - 1)
        anchors, covered = {}, set()
        for merged_range in sheet.merged_cells.ranges:
            left, top, right, bottom = range_boundaries(str(merged_range))
            ct, cl = max(top, table['top']), max(left, table['left'])
            cb, cr = min(bottom, visible_bottom), min(right, visible_right)
            if ct > cb or cl > cr:
                continue
            anchors[(ct, cl)] = (cb - ct + 1, cr - cl + 1)
            covered.update((r, c) for r in range(ct, cb + 1) for c in range(cl, cr + 1) if (r, c) != (ct, cl))
        html_rows = []
        for r in range(table['top'], visible_bottom + 1):
            cells = []
            for c in range(table['left'], visible_right + 1):
                if (r, c) in covered:
                    continue
                rs, cs = anchors.get((r, c), (1, 1))
                val = escape('' if sheet.cell(r, c).value is None else str(sheet.cell(r, c).value)).replace('\n', '<br>')
                cls = 'eform-header' if r <= table['header_bottom'] else 'eform-value'
                cells.append(f'<td class="{cls}" rowspan="{rs}" colspan="{cs}">{val}</td>')
            html_rows.append('<tr>' + ''.join(cells) + '</tr>')
        rng = f"{get_column_letter(table['left'])}{table['top']}:{get_column_letter(table['right'])}{table['bottom']}"
        style = '''<style>.eform-preview{border-collapse:collapse;margin:6px 0 18px;font-family:Arial;font-size:11px}.eform-preview td{border:1px solid #64748b;padding:4px;max-width:180px}.eform-header{background:#dbeafe;font-weight:600;text-align:center}.eform-value{background:#f8fafc}</style>'''
        display(HTML(f"{style}<h4>{escape(record['split'])} · {escape(record['variant_type'])} · {rng} ({len(record['tables'])} bảng)</h4><div>{escape(record['source_file'])}</div><table class='eform-preview'>{''.join(html_rows)}</table>"))
    finally:
        wb.close()

for vtype in ('original', 'unmerged', 'borderless', 'no_code_row', 'shifted_pos', 'two_tables', 'combo_unmerged_borderless_footer', 'no_table_guide_sheet'):
    cand = next((r for r in records if r.get('variant_type') == vtype), None)
    if cand:
        show_table_preview(cand)

STT,,Số VBQPPL đã được ban hành tại cấp xã,,,"Số VBQPPL đã được ban hành, được đánh giá tác động giới tại cấp tỉnh",,,,
,,Tổng số,Chia theo tên loại VBQPPL,,Tổng số,Chia theo tên loại VBQPPL,,,
,,,Nghị quyết của HĐND,Quyết định của UBND,,Nghị quyết của HĐND,,Quyết định của UBND,Quyết định của Chủ tịch UBND
,,,,,,Tổng số,Trong đó: Số VBQPPL có đánh giá tác động giới,,
A,B,(1),(2),(3),(4),(5),(6),(7),(8)
,Tổng số trên địa bàn tỉnh,22,0,22,0,,,,
I,Tại cấp tỉnh,-,-,-,0,,,,
II,Tại cấp xã,22,0,22,-,-,-,-,-


STT,,Số VBQPPL đã được ban hành tại cấp xã,,,"Số VBQPPL đã được ban hành, được đánh giá tác động giới tại cấp tỉnh",,,,
,,Tổng số,Chia theo tên loại VBQPPL,,Tổng số,Chia theo tên loại VBQPPL,,,
,,,Nghị quyết của HĐND,Quyết định của UBND,,Nghị quyết của HĐND,,Quyết định của UBND,Quyết định của Chủ tịch UBND
,,,,,,Tổng số,Trong đó: Số VBQPPL có đánh giá tác động giới,,
A,B,(1),(2),(3),(4),(5),(6),(7),(8)
,Tổng số trên địa bàn tỉnh,212,28,184,43,6,0,36,1
I,Tại cấp tỉnh,-,-,-,43,6,0,36,1
II,Tại cấp xã,212,28,184,-,-,-,-,-
